# Infer-19 — Analyse de survie / fiabilite bayesienne : inferer le temps jusqu'a un evenement

> **Corpus bayesien Infer.NET.** Ce notebook (19e du corpus) ouvre la famille des
> **modeles de duree** (time-to-event) : la quantite centrale n'est plus une moyenne ou un compte,
> mais une **fonction de survie** `S(t) = P(T > t)` — probabilite qu'un composant, un patient ou
> un client survive au-dela de l'instant `t`. Il complete la famille « sequences dans le temps »
> ([Infer-14 HMM](Infer-14-Sequences.ipynb), [Infer-17 Kalman](Infer-17-Kalman-Filter.ipynb),
> [Infer-18 Change-Point](Infer-18-Change-Point.ipynb)) par le cas ou l'observation n'est pas un
> etat recurrent mais un **delai unique** avant evenement.

**Plan.** (1) Pourquoi un modele dedie aux durees. (2) Le modele exponentiel (cas conjugue).
(3) Fonction de survie predictive en forme fermee. (4) Le modele de Weibull via une astuce de
transformee. (5) Selection de la forme `k`. (6) Trois exercices (censure, regression AFT,
comparaison de modeles).

## 1. Motivation : pourquoi pas une gaussienne ?

Un ingenieur fiabilite observe les **durees de vie** (en heures) de `N` composants identiques
soumis a un test. La grandeur qui l'interesse est : *« quelle probabilite qu'un composant
fonctionne encore apres 1500 h ? »* — c'est-a-dire `S(1500) = P(T > 1500)`.

Modeliser `T` par une gaussienne est inadequat pour trois raisons :

1. **Le temps est positif.** Une gaussienne attribue une masse non nulle aux durees negatives.
2. **La distribution est asymetrique a droite.** Quelques composants vivent tres longtemps
   (longue queue), la moyenne depasse la mediane.
3. **Ce qui importe est la queue**, pas le centre : `S(t)` dans la region ou peu de composants
   sont encore en vie.

Deux lois canoniques repondent a ces contraintes :

- **Exponentielle** `T ~ Exp(lambda)` : taux de defaillance **constant** dans le temps
  (memoire, usure nulle). Un seul parametre `lambda > 0` (le taux).
- **Weibull** `T ~ Weibull(k, eta)` : taux de defaillance qui **varie** — `k < 1` (mortalite
  infantile, le composant se rodit), `k = 1` (retombe sur l'exponentielle), `k > 1` (usure,
  le risque croit avec l'age).

Le defi d'inference : estimer `lambda`, ou le couple `(k, eta)`, a partir de durees observees,
puis **propager l'incertitude** jusqu'a `S(t)` — avec un intervalle de credibilite, pas un
chiffre unique. C'est exactement ce que le calcul bayesien via Infer.NET fournit.

In [1]:
#r "nuget: Microsoft.ML.Probabilistic"
#r "nuget: Microsoft.ML.Probabilistic.Compiler"
// restore Infer.NET -- isole dans sa propre cellule (convention de la serie)

The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages Microsoft.ML.Probabilistic, 0.4.2504.701 Microsoft.ML.Probabilistic.Compiler, 0.4.2504.701

Configuration des espaces de noms Infer.NET. On retrouve le moteur **EP** (Expectation
Propagation) et les helpers. On ajoute une graine fixee pour que les durees synthetiques soient
reproductibles (le test porte sur l'inference, pas sur le tirage).

In [2]:
using Microsoft.ML.Probabilistic;
using Microsoft.ML.Probabilistic.Distributions;
using Microsoft.ML.Probabilistic.Models;
using Microsoft.ML.Probabilistic.Algorithms;
using Range = Microsoft.ML.Probabilistic.Models.Range;   // desambiguise vs System.Range
using System;
using System.Linq;

var rand = new Random(42);

// Helper : tirage uniforme et exponentiel reproductibles.
double Uniform() => rand.NextDouble();                                   // U ~ Uniform(0,1)
double SampleExponential(double rate) => -Math.Log(Uniform()) / rate;     // T ~ Exp(rate)
double SampleWeibull(double shape, double scale)                         // T ~ Weibull(k, eta)
    => scale * Math.Pow(-Math.Log(Uniform()), 1.0 / shape);

Console.WriteLine("Infer.NET charge. Helpers Uniform / SampleExponential / SampleWeibull definis.");

Infer.NET charge. Helpers Uniform / SampleExponential / SampleWeibull definis.


## 2. Un jeu de donnees synthetique (verite connue)

On simule les durees de vie de `N = 60` composants dont le **vrai** taux de defaillance est
`lambda_true = 1/1000` (duree moyenne caracteristique 1000 h). Connaissant la verite, on pourra
verifier que le postieur la recouvre. On garde egalement un jeu **Weibull** avec usure
(`k = 1.8`) pour la section 4.

In [3]:
const int N = 60;
const double lambdaTrue = 1.0 / 1000.0;   // duree moyenne 1000 h

double[] lifetimesExp = Enumerable.Range(0, N)
    .Select(_ => SampleExponential(lambdaTrue)).ToArray();

// Second jeu : Weibull avec usure (k=1.8), meme duree caracteristique (eta=1000).
const double kTrue = 1.8, etaTrue = 1000.0;
double[] lifetimesWei = Enumerable.Range(0, N)
    .Select(_ => SampleWeibull(kTrue, etaTrue)).ToArray();

Console.WriteLine($"Durees Exponentiel : N={N}, moyenne observee={lifetimesExp.Average():F1} h (attendu ~1000)");
Console.WriteLine($"Durees Weibull      : N={N}, moyenne observee={lifetimesWei.Average():F1} h (k=1.8 -> queue plus courte)");

Durees Exponentiel : N=60, moyenne observee=1262,9 h (attendu ~1000)


Durees Weibull      : N=60, moyenne observee=1041,0 h (k=1.8 -> queue plus courte)


## 3. Le modele exponentiel : le cas conjugue

Le modele le plus simple : `T_i ~ Exp(lambda)` pour chaque composant, avec un *a priori*
**Gamma** sur le taux `lambda`. Le prior Gamma est **conjugue** a la vraisemblance exponentielle :
EP renvoie donc un postieur exact, lui-meme Gamma.

**Lien de conjugaison.** Si `lambda ~ Gamma(a0, b0)` (parametre forme `a0`, parametre **taux**
`b0`, donc moyenne `a0/b0`) et `T_i ~ Exp(lambda)` pour `i = 1..N`, alors le postieur est
`Gamma(a0 + N, b0 + somme(T_i))`. On retrouve ainsi, a la main, ce que le moteur calcule :
chaque observation ajoute `1` a la forme et sa duree `T_i` au taux. Le prior faible
`Gamma(0.001, 0.001)` laisse les donnees parler.

In [4]:
// Modele exponentiel conjugue sur le jeu de durees exponentielles.
Range item = new Range(N);
var lambda = Variable.GammaFromShapeAndRate(0.001, 0.001).Named("lambda");   // prior vague sur le taux
var T = Variable.Array<double>(item).Named("T");
T[item] = Variable.GammaFromShapeAndRate(1.0, lambda).ForEach(item);   // shape=1 => Exponentielle(lambda)
T.ObservedValue = lifetimesExp;

var engine = new InferenceEngine { Algorithm = new ExpectationPropagation() };
Gamma lambdaPost = engine.Infer<Gamma>(lambda);

Console.WriteLine("=== Posterieur du taux lambda (modele exponentiel) ===");
Console.WriteLine($"  Forme A   = {lambdaPost.Shape:F3}");
Console.WriteLine($"  Taux  B   = {lambdaPost.Rate:F3}");
Console.WriteLine($"  Moyenne   = {lambdaPost.GetMean():F6}  (vrai lambda = {lambdaTrue:F6})");
Console.WriteLine($"  Ecart-type= {Math.Sqrt(lambdaPost.GetVariance()):F6}");

Compiling model...

done.


=== Posterieur du taux lambda (modele exponentiel) ===


  Forme A   = 60,001


  Taux  B   = 75774,766


  Moyenne   = 0,000792  (vrai lambda = 0,001000)


  Ecart-type= 0,000102


### Lecture du postérieur : la conjugaison se vérifie chiffres en main

Le postérieur `Gamma(A, B)` renvoyé par EP coincide avec la formule de conjugaison :

- `A = 60,001 = 0,001 + N` — la forme est l'a priori plus le nombre d'observations, **au centieme pres**.
- `B = 75 774,77 = 0,001 + Σ T_i` — le taux est l'a priori plus la somme des durees
  (or `Σ T_i / 60 = 1262,9 h`, exactement la moyenne observee).

La moyenne postérieure `A/B = 7,92e-4` est **identique a l'estimateur du maximum de vraisemblance**
`1 / moyenne(T_i)`, parce que le prior `Gamma(0,001 ; 0,001)` est negligible devant 60 donnees. Elle
est legerement inferieure au vrai `lambda = 1e-3` non par biais du modele, mais parce que **cet
echantillon** a tire une moyenne de 1262,9 h (queue haute) au lieu de 1000 : avec `N = 60`,
l'intervalle de confiance sur la moyenne est large, et le postérieur le reflete fidelement
(ecart-type `~1,0e-4`, soit un coefficient de variation de 13 %). C'est precisement l'interet du
bayesien : on recupere non un chiffre, mais une **distribution** dont on propagera l'incertitude
jusqu'a la fonction de survie.

## 4. Fonction de survie predictive : une forme fermee

La question de l'ingenieur — *« probabilite de survivre au-dela de 1500 h »* — se traduit par
la **survie predictive** : on moyenne la survie `exp(-lambda * t)` sur le postieur de `lambda`.

**Forme fermee.** Si `lambda ~ Gamma(A, B)`, alors par transformee de Laplace :

```
S(t) = E_lambda[ exp(-lambda * t) ] = ( B / (B + t) )^A
```

Pas besoin d'echantillonner : pour chaque `t`, on branche les moments du postieur Gamma et on
obtient `S(t)` **exactement**, avec son incertitude qui decroit quand le postieur se concentre.
On trace `S(t)` sur `t ∈ [0, 3000]` et on le compare a la courbe empirique (Kaplan-Meier simplifiee,
ici sans censure : fraction d'observations superieures a `t`).

In [5]:
#load "SvgChartHelper.cs"
// Survie predictive (forme fermee) + comparaison empirique.
double A = lambdaPost.Shape, B = lambdaPost.Rate;
double Sbayes(double t) => Math.Pow(B / (B + t), A);
double Semp(double t, double[] data) => data.Count(d => d > t) / (double)data.Length;

// Tableau : S(t) bayesienne vs empirique a des horizons choisis.
Console.WriteLine("=== Fonction de survie S(t) : bayesienne vs empirique ===");
Console.WriteLine($"{"t (h)",8}  {"S_bayes",8}  {"S_empir",8}");
foreach (var t in new[] { 0.0, 250, 500, 1000, 1500, 2000, 3000 })
    Console.WriteLine($"{t,8:F0}  {Sbayes(t),8:F3}  {Semp(t, lifetimesExp),8:F3}");

// Courbe continue S(t) : trace Plotly comparant le modele bayesien aux donnees empiriques.
int NP = 80;
var tGrid = Enumerable.Range(0, NP + 1).Select(j => j * 3000.0 / NP).ToArray();
SvgChartHelper.Overlay(
    "Fonction de survie S(t) : bayesienne vs empirique",
    "t (heures)", "S(t)",
    new[] {
        new SvgSeries("Bayesienne (modele exponentiel)",
            tGrid, tGrid.Select(Sbayes).ToArray(), TraceStyle.Line, "#2a6dba"),
        new SvgSeries("Empirique (donnees)",
            tGrid, tGrid.Select(t => Semp(t, lifetimesExp)).ToArray(),
            TraceStyle.Markers, "#e8743b"),
    })

=== Fonction de survie S(t) : bayesienne vs empirique ===


   t (h)   S_bayes   S_empir


       0     1,000     1,000


     250     0,821     0,900


     500     0,674     0,700


    1000     0,455     0,450


    1500     0,308     0,317


    2000     0,209     0,183


    3000     0,097     0,100


Fonction de survie S(t) : bayesienne vs empirique 0.043 0.296 0.549 0.801 1.054 0 750 1500 2250 3000 t (heures) S(t) <polyline points='52,55.643 61.4,67.331 70.8,78.673 80.2,89.677 89.6,100.354 99,110.715 108.4,120.768 117.8,130.523 127.2,139.989 136.6,149.174 146,158.088 155.4,166.738 164.8,175.132 174.2,183.278 183.6,191.183 193,198.854 202.4,206.3 211.8,213.525 221.2,220.537 230.6,227.343 240,233.948 249.4,240.359 258.8,246.581 268.2,252.619 277.6,258.481 287,264.17 296.4,269.691 305.8,275.051 315.2,280.254 324.6,285.303 334,290.205 343.4,294.963 352.8,299.581 362.2,304.065 371.6,308.417 381,312.642 390.4,316.743 399.8,320.725 409.2,324.59 418.6,328.342 428,331.985 437.4,335.521 446.8,338.955 456.2,342.288 465.6,345.525 475,348.667 484.4,351.717 493.8,354.679 503.2,357.555 512.6,360.347 522,363.058 531.4,365.691 540.8,368.247 550.2,370.729 559.6,373.139 569,375.479 578.4,377.751 587.8,379.958 597.2,382.101 606.6,384.181 616,386.202 625.4,388.164 634.8,390.07 644.2,391.92 653.6,393.718 663,395.463 672.4,397.158 681.8,398.804 691.2,400.403 700.6,401.956 710,403.464 719.4,404.929 728.8,406.351 738.2,407.733 747.6,409.075 757,410.379 766.4,411.645 775.8,412.875 785.2,414.07 794.6,415.23 804,416.357' fill='none' stroke='#2a6dba' stroke-width='2'/> Bayesienne (modele exponentiel) Empirique (donnees)

### Lecture : la question de l'ingenieur a maintenant une reponse quantifiee

La question *« probabilite qu'un composant survive au-dela de 1500 h »* admet la reponse
`S(1500) ≈ 0,31` : environ un composant sur trois est encore en vie a 1500 h. Deux points :

- **Accord bayesien / empirique.** Les deux courbes se suivent (ecart maximal ~0,08 a `t = 250`)
  mais de nature differente : la bayesienne est **parametrique et lisse** (elle croit au modele
  exponentiel et projette sa forme), l'empirique est **en escalier** (fraction brute des durees
  superieures a `t`). Leur proximite valide que l'exponentielle decrit bien ce jeu.
- **Forme fermee exacte.** `S(t) = (B/(B+t))^A` n'est pas une approximation Monte-Carlo : c'est
  la transformee de Laplace du postieur Gamma, calculee en un seul coup. On peut donc donner un
  intervalle de credibilite sur `S(t)` en tirant des `lambda` du postieur — l'incertitude sur le
  taux se propage gratuitement jusqu'a la queue, la ou l'enjeu fiabilite se trouve.

## 5. Le modele de Weibull : une astuce de transformee

L'exponentielle suppose un taux **constant** (ni rodage, ni usure). La loi de **Weibull**
generalise : `T ~ Weibull(k, eta)` avec fonction de survie `S(t) = exp(-(t/eta)^k)`.

Inferer directement la **forme** `k` par EP est delicat (la vraisemblance de Weibull n'est
conjuguee a aucun prior usuel). Mais si l'on fixe `k`, une **transformee** ramene le probleme au
cas conjugue :

```
U_i = T_i^k  ==>  U_i ~ Exp(rate = eta^{-k})
```

On infere donc le taux transforme `r = eta^{-k}` avec un prior Gamma (conjugue), puis on remonte
a `eta = r^{-1/k}`. La survie predictive devient `S(t) = (B / (B + t^k))^A` (meme forme fermee,
avec `t` remplace par `t^k`). La section suivante choisira le meilleur `k` par balayage.

In [6]:
// Weibull a forme k fixee : transformee U = T^k ~ Exp(r), r = eta^{-k}.
double InferWeibullRate(double k, double[] data, out Gamma rPost)
{
    Range it = new Range(data.Length);
    var r = Variable.GammaFromShapeAndRate(0.001, 0.001).Named("r_" + k.ToString("F1"));
    var U = Variable.Array<double>(it).Named("U_" + k.ToString("F1"));
    U[it] = Variable.GammaFromShapeAndRate(1.0, r).ForEach(it);   // shape=1 => Exponentielle(r)
    U.ObservedValue = data.Select(t => Math.Pow(t, k)).ToArray();
    var eng = new InferenceEngine { Algorithm = new ExpectationPropagation() };
    rPost = eng.Infer<Gamma>(r);
    return rPost.GetMean();
}

double kFixed = 1.8;   // (on retrouvera ce k par balayage a la section 6)
double rMean = InferWeibullRate(kFixed, lifetimesWei, out Gamma rPost);
double etaMean = Math.Pow(rMean, -1.0 / kFixed);

Console.WriteLine("=== Modele Weibull (k=1.8 fixe) sur le jeu Weibull ===");
Console.WriteLine($"  r = eta^-k      = {rMean:E3}");
Console.WriteLine($"  eta (back-out)  = {etaMean:F1} h  (vrai eta = {etaTrue:F1})");
Console.WriteLine($"  Survie a 1000 h = {Math.Pow(rPost.Rate/(rPost.Rate + Math.Pow(1000,kFixed)), rPost.Shape):F3}");

Compiling model...

done.


=== Modele Weibull (k=1.8 fixe) sur le jeu Weibull ===


  r = eta^-k      = 2,926E-006


  eta (back-out)  = 1186,7 h  (vrai eta = 1000,0)


  Survie a 1000 h = 0,482


### Lecture : la transformee ramene Weibull au cas conjugue

L'astuce `U = T^k ~ Exp(rate = eta^{-k})` fonctionne : EP infere un postérieur Gamma sur `r` sans
aucune fragilite (pas d'inference directe sur la forme). On remonte `eta = r^{-1/k} ≈ 1187 h`,
a comparer au vrai `eta = 1000 h`. La surestimation vient, comme pour l'exponentiel, de
l'echantillon : sa moyenne (1041 h) est un peu haute, et avec `k = 1,8` on a
`moyenne = eta · Γ(1 + 1/k) ≈ eta · 0,89`, d'ou un `eta` implique par les donnees autour de 1170 h.

La survie estimee `S(1000) ≈ 0,48` (contre `e^{-1} ≈ 0,37` pour le vrai `Weibull(1,8 ; 1000)`)
reflete cette meme surestimation de `eta`. Ce n'est pas un defaut de la methode mais du bruit
d'echantillonnage a `N = 60` ; l'apport pedagogique est ailleurs : **toute la richesse de Weibull
(usure, rodage) est accessible sans payer le cout d'une inference EP sur la forme**, des lors que
l'on fixe `k` (ou qu'on le choisit par balayage, section suivante).

## 6. Selection de la forme k : balayage par log-vraisemblance predictive

Comment choisir `k` sans payer le cout d'une inference EP sur la forme ? On **balaye** `k` sur
une grille et, pour chaque `k`, on calcule la **log-vraisemblance predictive** (leave-one-out
approchee) du jeu observe : pour chaque duree `T_i`, la densite Weibull evaluee avec le postieur
entraene sur les autres. Le `k` qui maximise cette score est le meilleur compromis biais/variance.

Sur le jeu exponentiel (taux constant), `k ≈ 1` doit gagner ; sur le jeu Weibull `k = 1.8`,
c'est `k ≈ 1.8` qui doit gagner. C'est un test de coherence interne.

In [7]:
// Balayage de k : pour chaque k, log-vraisemblance predictive (LOO approchee).
double ShapeScore(double k, double[] data)
{
    // Posterieur du taux transforme r sur TOUT le jeu, densite Weibull evaluee en chaque point.
    // (approximation LOO : le postieur est peu sensible a un seul point quand N est grand.)
    double rMean = InferWeibullRate(k, data, out Gamma rPost);
    double A_ = rPost.Shape, B_ = rPost.Rate;
    // Densite de T_i sous Weibull(k, eta) avec eta = r^{-1/k}, r ~ Gamma(A,B).
    // On evalue la log-densite marginale approchee a la moyenne post.: r_hat = A/B.
    double rHat = A_ / B_;
    double eta = Math.Pow(rHat, -1.0 / k);
    double lp = 0.0;
    foreach (var t in data)
    {
        double z = t / eta;
        lp += Math.Log(k) - k * Math.Log(eta) + (k - 1) * Math.Log(t) - Math.Pow(z, k);
    }
    return lp;
}

Console.WriteLine("=== Selection de la forme k (log-vraisemblance predictive) ===");
Console.WriteLine($"{"k",5}  {"score Exp",12}  {"score Weibull",14}");
var ks = new[] { 0.8, 1.0, 1.2, 1.5, 1.8, 2.0, 2.5 };
double bestExp = double.NegativeInfinity, bestWei = double.NegativeInfinity;
double kExp = 1, kWei = 1;
foreach (var k in ks)
{
    double se = ShapeScore(k, lifetimesExp);
    double sw = ShapeScore(k, lifetimesWei);
    Console.WriteLine($"{k,5:F1}  {se,12:F1}  {sw,14:F1}");
    if (se > bestExp) { bestExp = se; kExp = k; }
    if (sw > bestWei) { bestWei = sw; kWei = k; }
}
Console.WriteLine($"\n  k* (jeu exponentiel) = {kExp:F1}  (attendu ~1.0)");
Console.WriteLine($"  k* (jeu Weibull)     = {kWei:F1}  (attendu ~1.8)");

=== Selection de la forme k (log-vraisemblance predictive) ===


    k     score Exp   score Weibull


Compiling model...

done.


Compiling model...

...

Compiling model...

done.


  2,5        -526,1          -475,7



  k* (jeu exponentiel) = 1,2  (attendu ~1.0)


  k* (jeu Weibull)     = 1,8  (attendu ~1.8)


### Lecture : le balayage recupere la vraie forme et distingue les deux regimes

Le test de coherence reussit sur les deux jeux :

- **Jeu Weibull** (`k = 1,8`) : le score est **minimal en `k = 1,8`** (`-466,7`), c'est-a-dire
  exactement la valeur vraie. Le pic est net (`k = 1,5` et `k = 2,0` sont deja moins bons).
  La methode retrouve la signature d'usure.
- **Jeu exponentiel** (`k = 1`) : le meilleur est `k = 1,2` (`-486,8`), mais `k = 1,0` (`-488,5`)
  est a moins de 2 unites : **ex aequo dans le bruit d'echantillonnage**. Leger tilt vers `1,2`
  parce que cet echantillon exponentiel n'etait pas un exponentiel parfait. On conclut donc
  raisonnablement `k ≈ 1` (taux constant), ce qui est la bonne reponse.

Cette approche par **grille** evite l'inference EP directe sur la forme (notoirement instable) au
prix de quelques compilations de modele (7 ici, chacune instantanee). C'est le compromis
operationnel : robustesse contre temps de calcul, dans l'esprit de la regle « realiser le moteur,
pas le contourner ».

## 7. Exercices

Les trois exercices ci-dessous etendent le modele de duree. Ils sont laisses **a completer**
(stubs sans erreur) — le notebook s'execute de bout en bout meme non rempli.

### Exercice 1 — Censure a droite (donnees tronquees)

En fiabilite reelle, beaucoup de composants **survivent** a la fin du test : on n'observe pas
leur duree exacte `T_i`, seulement qu'elle depasse la duree du test `c_i`. C'est la **censure a
droite**. La contribution de vraisemblance d'un point censure n'est pas la densite `f(c_i)` mais
la survie `S(c_i) = exp(-lambda * c_i)`.

**Indice :** separez indices observes / censures. Pour les observes, branchez `T_i ~ Exp(lambda)`
comme en section 3. Pour les censures, ajoutez un facteur de survie : en Infer.NET on peut
exprimer `exp(-lambda * c_i)` comme la probabilite qu'une variable latente exponentielle de taux
`lambda` soit superieure a `c_i` (via `Variable.ConstrainPositive` sur `c_i - T_i`, ou en
modelisant un indicateur d'evenement). Comparez le postieur obtenu avec/sans censure.

In [8]:
// Exercice 1 : censure a droite (a completer)
// TODO etudiant : repartir indices observes/censures, ajouter le facteur S(c_i) pour les censures.
Console.WriteLine("Exercice 1 a completer : modele exponentiel avec censure a droite.");

Exercice 1 a completer : modele exponentiel avec censure a droite.


### Exercice 2 — Regression de temps accelere (AFT)

La duree depend d'une covariable (temperature, contrainte). Modele **AFT** (Accelerated Failure
Time) : le taux devient specifique au composant, `lambda_i = lambda0 * exp(beta * x_i)` ou `x_i`
est la covariable centree et `beta` le coefficient a inferer.

**Indice :** declarez `lambda0 ~ Gamma` et `beta ~ Gaussian(0, grande)`, puis bouclez sur les
composants avec `Variable.Exponential(lambda0 * Variable.Exp(beta * x[i]))`. Inferer `beta`
donne l'effet de la contrainte sur la duree de vie (signe et amplitude). Attention : le produit
dans le taux rend le postieur non conjugue — EP le traite quand meme (approximation).

In [9]:
// Exercice 2 : regression AFT (a completer)
// TODO etudiant : lambda_i = lambda0 * exp(beta * x_i), inferer beta.
Console.WriteLine("Exercice 2 a completer : regression de temps accelere (AFT).");

Exercice 2 a completer : regression de temps accelere (AFT).


### Exercice 3 — Exponentiel vs Weibull : facteur de Bayes

Sur un meme jeu de durees, comparer le modele exponentiel (`k = 1`) au modele Weibull (`k` libre)
via un **facteur de Bayes** (rapport des vraisemblances marginales). Le verdict depend de la
taille `N` et du vrai `k`.

**Indice :** la section 6 calcule deja un score par `k`. Generalisez-le en vraisemblance
marginale (intégrez sur le postieur de `r`, pas juste evaluez a la moyenne), puis formez le
rapport `BF = p(D | Weibull) / p(D | exponentiel)`. A partir de quel `N` le Weibull `k = 1.8`
est-il prefere de facon decisive (`log BF > 5`) ?

In [10]:
// Exercice 3 : facteur de Bayes exponentiel vs Weibull (a completer)
// TODO etudiant : vraisemblance marginale par k, rapport BF, seuil en N.
Console.WriteLine("Exercice 3 a completer : facteur de Bayes exponentiel vs Weibull.");

Exercice 3 a completer : facteur de Bayes exponentiel vs Weibull.


## Conclusion

L'analyse de survie complete la famille des modeles temporels du corpus Infer :

| Notebook | Nature du temps | Quantite inferee |
|----------|-----------------|------------------|
| [Infer-14](Infer-14-Sequences.ipynb) (HMM) | discret, recurrent | trajectoire d'etats caches |
| [Infer-17](Infer-17-Kalman-Filter.ipynb) (Kalman) | continu, recurrent | etat filtre pas-a-pas |
| [Infer-18](Infer-18-Change-Point.ipynb) (Change-Point) | continu, a rupture | instant d'un changement de regime |
| **Infer-19 (Survie)** | **continu, duree unique** | **fonction de survie S(t)** |

**A retenir.**

- Le temps jusqu'a un evenement se modelise par une loi **positive asymetrique** (exponentielle,
  Weibull), pas une gaussienne — ce qui compte est la queue, i.e. `S(t)`.
- Le cas **exponentiel** est conjugue : prior Gamma sur le taux, postieur Gamma exact. La survie
  predictive se ramene a une **forme fermée** `(B/(B+t))^A` (transformee de Laplace du Gamma).
- Le cas **Weibull** se ramene a l'exponentiel par **transformee** `U = T^k` des que `k` est fixe,
  evitant la fragilite de EP sur la forme. On choisit `k` par balayage (log-vraisemblance
  predictive), non par inference directe.
- La censure, la regression AFT et la selection par facteur de Bayes sont les prolongements
  naturels (exercices).

**Pour aller plus loin.** La censure a gauche/par intervalle, les modeles a risques concurrents
(competing risks) et les modeles de fragilite partagee (frailty, analogue hierarchique de
[Infer-12](Infer-12-Modeles-Hierarchiques.ipynb) applique a la survie) sont les extensions
classiques au-dela de ce notebook.

---

### References

- Gelman A. et al., *Bayesian Data Analysis* (3e ed.), chap. « Survival models ».
- Lawless J. F., *Statistical Models and Methods for Lifetime Data* — reference pour Weibull/AFT.
- Infer.NET documentation : `Variable.GammaFromShapeAndRate` (prior sur le taux ; shape=1 donne
  l'exponentielle), `Variable.Weibull`.
- Relation a la serie : [Infer-12](Infer-12-Modeles-Hierarchiques.ipynb) (modeles hierarchiques),
  [Infer-18](Infer-18-Change-Point.ipynb) (rupture — ou le taux change brusquement).
